# S47_05 — End-to-End RAG Pipeline

This notebook assembles all RAG components into a working pipeline using LangChain — the most common RAG orchestration framework.

## LangChain RAG with ChromaDB

In [ ]:
# pip install langchain langchain-anthropic langchain-community chromadb sentence-transformers
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- 1. Build the knowledge base ---
raw_docs = [
    Document(page_content="""
    Python is a high-level, general-purpose programming language. Its design philosophy emphasizes 
    code readability with the use of significant indentation. Python is dynamically typed and 
    garbage-collected. It supports multiple programming paradigms, including structured, 
    object-oriented and functional programming. It is often described as a batteries included 
    language due to its comprehensive standard library.
    """, metadata={'source': 'python_intro.txt'}),
    
    Document(page_content="""
    NumPy is the fundamental package for scientific computing with Python. It provides a 
    multidimensional array object, various derived objects (such as masked arrays and matrices), 
    and an assortment of routines for fast operations on arrays, including mathematical, 
    logical, shape manipulation, sorting, selecting, I/O, discrete Fourier transforms, 
    basic linear algebra, basic statistical operations, random simulation and much more.
    """, metadata={'source': 'numpy_intro.txt'}),
    
    Document(page_content="""
    pandas is a fast, powerful, flexible and easy to use open source data analysis and 
    manipulation tool, built on top of the Python programming language. It offers data 
    structures and operations for manipulating numerical tables and time series. The name 
    pandas is derived from the term panel data, an econometrics term for data sets that 
    include observations over multiple time periods for the same individuals.
    """, metadata={'source': 'pandas_intro.txt'}),
    
    Document(page_content="""
    scikit-learn is a free software machine learning library for Python. It features various 
    classification, regression and clustering algorithms including support-vector machines, 
    random forests, gradient boosting, k-means and DBSCAN, and is designed to interoperate 
    with the Python numerical and scientific libraries NumPy and SciPy.
    """, metadata={'source': 'sklearn_intro.txt'}),
]

# --- 2. Chunk ---
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
print(f'Split {len(raw_docs)} docs into {len(chunks)} chunks')

In [ ]:
# --- 3. Embed and store ---
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    # persist_directory='./rag_chroma_db',  # uncomment to persist to disk
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3},
)

# Test retrieval
test_results = retriever.invoke('What is pandas used for?')
print(f'Retrieved {len(test_results)} chunks')
for r in test_results:
    print(f'  [{r.metadata["source"]}]: {r.page_content[:80]}...')

In [ ]:
# --- 4. Build the RAG chain ---
llm = ChatAnthropic(model='claude-haiku-4-5-20251001', max_tokens=512)

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the provided context.
If the context doesn't contain enough information, say so — do not guess.
Cite the source document(s) at the end of your answer.

Context:
{context}

Question: {question}
""")

def format_docs(docs):
    return '\n\n'.join(
        f'[Source: {doc.metadata["source"]}]\n{doc.page_content}'
        for doc in docs
    )

# LCEL chain: question → {context, question} → prompt → llm → string
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Ask a question
answer = rag_chain.invoke('What is scikit-learn and what algorithms does it include?')
print(answer)

## Evaluating RAG quality

In [ ]:
# Simple RAG evaluation: faithfulness + relevance
import anthropic
import json

client = anthropic.Anthropic()

def evaluate_rag_answer(question, context, answer):
    """Score answer on faithfulness (grounded in context) and relevance."""
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=256,
        system='You evaluate RAG system answers. Respond ONLY with valid JSON.',
        messages=[{'role': 'user', 'content': f"""
Question: {question}

Context provided to the system:
{context[:500]}

System answer:
{answer[:500]}

Evaluate:
{{"faithfulness": {{"score": 1-5, "reason": str}}, "relevance": {{"score": 1-5, "reason": str}}}}
"""}],
    )
    return json.loads(msg.content[0].text)

test_q = 'What is scikit-learn?'
retrieved_context = format_docs(retriever.invoke(test_q))
test_answer = rag_chain.invoke(test_q)

eval_result = evaluate_rag_answer(test_q, retrieved_context, test_answer)
print(json.dumps(eval_result, indent=2))

## RAG failure modes and fixes

| Problem | Symptom | Fix |
|---------|---------|-----|
| Wrong chunks retrieved | Answer ignores the question | Improve chunking; try hybrid retrieval |
| Hallucination despite context | Answer contradicts sources | Stronger system prompt; constrain to context |
| Missing answer | "I don't know" when info exists | Smaller chunks; increase top-k |
| Slow pipeline | High latency | Cache embeddings; use faster embed model |
| Stale knowledge | Old information returned | Incremental upsert with timestamps |

## Advanced RAG patterns

- **Parent-child chunks**: store small chunks for retrieval, fetch parent chunk for LLM context
- **Hypothetical Document Embeddings (HyDE)**: generate a hypothetical answer, embed it, use that to retrieve
- **RAPTOR**: recursive tree-based summarization for multi-level retrieval
- **Corrective RAG**: evaluate retrieval quality and fall back to web search if poor

This completes S47. Next section: [S48_AI_Agents](../S48_AI_Agents/S48_01_what_are_agents.ipynb)